# 02 · Barrido paramétrico de carga

> **validación humana requerida** antes de usar resultados en ingeniería.

`mamut_engine.experiments.sweep` prepara copias del voladizo y llama a Rust
para cada variante. Python no implementa el análisis. Comprobamos Uy proporcional
a P y la forma cerrada con los parámetros del fixture. No añadimos métodos.
`schemas/engine-input/1.json` (línea 5) y `methods.rs` declaran Experimental solo
`nonlinearStatic`, `soilProfile` y `consolidation1d`; pDelta es Stable; ENG-BENCH-002;
Validated pendiente de ENG-VAL-006. Otras capacidades: validación DESCONOCIDO. Este notebook no valida otras capacidades.

Requiere la wheel local; ejecutar dentro del repo. No usa NumPy ni pandas.
Consulta la [guía](../docs/GUIA_USUARIO.md) para instalación y límites.

In [ ]:
from pathlib import Path
import math
import mamut_engine as me

relative = Path("crates/engine-api/fixtures/input/valid/cantilever-linear-static.json")
cwd = Path.cwd().resolve()
root = next((p for p in (cwd, *cwd.parents)
             if (p / relative).is_file() and (p / "bindings/python").is_dir()), None)
if root is None:
    raise RuntimeError("Ejecuta dentro del repositorio para localizar el fixture")
model = me.Model.from_json((root / relative).read_text(encoding="utf-8"))
print("Versión y contratos:", me.version())
descriptors = {d["workflow"]["method_id"]: d for d in me.capabilities()["method_descriptors"]}
status = descriptors.get("linearStatic", {}).get("status", "DESCONOCIDO")
print("linearStatic:", "experimental: " + status if status == "Experimental" else status)

In [ ]:
# Leer geometría, carga y propiedades del documento, sin constantes físicas nuevas.
geometry = model.data["model"]
member = geometry["members"][0]
base_id, tip_id = member["nodes"]
base_node = next(n for n in geometry["nodes"] if n["id"] == base_id)
tip_node = next(n for n in geometry["nodes"] if n["id"] == tip_id)
assert geometry["axes"] == "Z-up" and model.data["units"] == "SI"
assert base_node["dim"] == tip_node["dim"] == "D2"
assert base_node["coordinates"][1:] == tip_node["coordinates"][1:] == [0, 0]
assert tip_node["coordinates"][0] > base_node["coordinates"][0]
L = math.dist(base_node["coordinates"], tip_node["coordinates"])
section = next(s for s in geometry["sections"] if s["id"] == member["section"])
law = next(law for law in geometry["laws"] if law["name"] == member["law"])
I = next(p["value"] for p in section["properties"] if p["name"] == "I")
E = next(p["value"] for p in law["parameters"] if p["name"] == "E")
input_case = model.data["cases"][0]
load_index, tip_load = next((i, load) for i, load in enumerate(input_case["loads"])
                            if load["node"] == tip_id)
component_index, component = next((i, v) for i, v in enumerate(tip_load["values"])
                                if v["dof"] == "Uy")
P = component["value"]  # Componente con signo, N.
assert L > 0 and E > 0 and I > 0 and P != 0
print({"P_N": P, "L_m": L, "E_Pa": E, "I_m4": I})

In [ ]:
report = me.validate(model, lang="es")
print("Diagnósticos de validación:", report["diagnostics"])
assert not any(d["severity"] == "error" for d in report["diagnostics"])
results = me.run(model, lang="es")
assert results.status in ("ok", "warning") and not results.partial
print("Estado:", results.status, "Diagnósticos:", results.diagnostics)
document = results.to_dict()

## Perfil reproducible

Seleccionamos el caso por nombre y el nodo por ID antes de construir la ruta de
la métrica. Las variantes mantienen la topología; después verificamos también
que esa ruta apunta a la punta en cada salida. La etiqueta `m` declara la unidad
del campo, sin conversión. La carga original se multiplica por 0,5; 1 y 2.

In [ ]:
from mamut_engine.experiments import comparison_table, sweep

case_index, reference_case = next((i, c) for i, c in enumerate(document["cases"])
                                if c["case"] == input_case["name"])
tip_index, reference_tip = next((i, n) for i, n in enumerate(reference_case["nodes"])
                              if n["id"] == tip_id)
reference_uy = reference_tip["displacements"][1]
loads = [factor * P for factor in (0.5, 1.0, 2.0)]
profile = {
    "schema": "mamut.experiment/1", "name": "voladizo-carga", "version": 1,
    "parameters": {"P_N": {
        "path": ["cases", 0, "loads", load_index, "values", component_index, "value"],
        "values": loads,
    }},
    "metrics": {"Uy_punta": {
        "path": ["cases", case_index, "nodes", tip_index, "displacements", 1],
        "unit": "m",
    }},
}
original = model.to_json()
rows = sweep(model, profile)
assert model.to_json() == original
assert len(rows) == len(loads)

## Linealidad y forma cerrada

Cada variante debe pasar ambas comprobaciones: `Uy/P` constante frente a la
ejecución original, y `Uy = P·L³/(3·E·I)` frente a una expectativa independiente.
Las tres cargas son no nulas; `rel_tol=1e-9`, `abs_tol=0` cubre redondeo binary64
del caso pequeño sin extrapolar la tolerancia a otros modelos.
`sweep` conserva errores por variante: `raise_for_error()` impide ocultarlos.

In [ ]:
for row, load in zip(rows, loads):
    row.raise_for_error()
    assert row.status in ("ok", "warning") and row.result is not None
    assert row.parameters["P_N"] == load
    variant_case = row.result.to_dict()["cases"][case_index]
    assert variant_case["case"] == input_case["name"]
    assert variant_case["nodes"][tip_index]["id"] == tip_id
    actual = row.metrics["Uy_punta"]["value"]
    expected = load * L**3 / (3 * E * I)
    assert row.metrics["Uy_punta"]["unit"] == "m"
    assert math.isclose(actual / load, reference_uy / P, rel_tol=1e-9, abs_tol=0.0)
    assert math.isclose(actual, expected, rel_tol=1e-9, abs_tol=0.0)
    print({"P_N": load, "Uy_Rust_m": actual, "Uy_analitico_m": expected})

In [ ]:
for row in comparison_table(rows):
    print(row["scenario"], row["status"], row["input_sha256"], row["result_sha256"])
    print("Productor:", row["provenance"]["engine_version"],
          "Commit:", row["provenance"].get("git_commit") or "DESCONOCIDO")
# input_sha256 es JCS del input en Rust; result_sha256 cubre los bytes de results/1.